# Inspect validated benchmark results

Run benchmarks from the repository root after building the release extension:

```bash
make sync-bench
make bench BENCH_OUTPUT=/tmp/h3-duckdb.json
```

For a short harness check, pass `BENCH_ARGS='--threads 2 --rows 128 --list-rows 32 --polygon-rows 2 --warmups 1 --iterations 2'`.
Use `UV_NO_SYNC=1 uv run -m benchmarks python --output /tmp/h3-python.json` for Python callbacks, or `internal` for broad Polars timings.
See the [benchmark guide](../benchmarks/README.md) for full commands and timing contracts and the [measured report](../benchmarks/results/2026-11-25-large-data.md) for published evidence.

This notebook reads saved JSON; it does not run a competing timer or combine unlike workloads into an overall speedup. Set the path below to a completed run. Use the repository's `.venv` kernel.


In [ ]:
import json
from pathlib import Path

import polars as pl

results_path = Path("/tmp/h3-duckdb.json")
report = json.loads(results_path.read_text())
if report.get("status", "complete") != "complete":
    raise ValueError("Choose a complete benchmark run")
report["metadata"]


In [ ]:
measurements = pl.DataFrame(
    [
        {
            "operation": case["name"],
            "rows": case["rows"],
            "mode": result["mode"],
            "engine": result["engine"],
            "median_ms": result["median_seconds"] * 1000,
            "min_ms": result["min_seconds"] * 1000,
            "max_ms": result["max_seconds"] * 1000,
            "samples": len(result["runs"]),
        }
        for case in report["cases"]
        for result in case["results"]
    ]
)
measurements.sort("operation", "mode", "engine")


Compare individual operations within the same timing mode and environment. Identity projections are controls, not H3 speedups. Tiny smoke-run timings are not performance evidence.

For before/after comparisons, use the shared comparison tool, which checks compatibility:

```bash
UV_NO_SYNC=1 uv run -m benchmarks compare /tmp/h3-before.json /tmp/h3-after.json
```

For a paired run containing a saved baseline, pass the same file twice and add `--baseline-engine baseline`. Raw samples remain in the JSON; published reports link to checksummed archives rather than committing every run.
